In [2]:
# MÓDULO 1 — PREPARAÇÃO DO AMBIENTE E DEFINIÇÃO DO RECORTE
# Define as temporadas, as pastas e as bibliotecas utilizadas.

from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    leaguedashplayerbiostats,
    shotchartdetail
)
from requests.exceptions import RequestException
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import time
import sys

temporadas = [
    f"{ano}-{str(ano + 1)[-2:]}"
    for ano in range(2015, 2025)
]

pasta_jogadores = Path("dados/jogadores_por_temporada")
pasta_arremessos = Path("dados/arremessos")
pasta_perfis = Path("dados/perfis_por_temporada")
pasta_auditorias = Path("dados/auditorias")

for pasta in [pasta_jogadores, pasta_arremessos, pasta_perfis]:
    pasta.mkdir(parents=True, exist_ok=True)

print("Python utilizado:", sys.executable)
print("Temporadas:", temporadas)

Python utilizado: c:\Users\felip\Desktop\teste-nba\.venv\Scripts\python.exe
Temporadas: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


In [5]:
# MÓDULO 2 — COLETA DAS ESTATÍSTICAS DOS JOGADORES POR TEMPORADA
# Reutiliza arquivos existentes ou consulta a API com até três tentativas.
# Valida a estrutura e as chaves da base que servirá de referência.

tabelas_jogadores = []

colunas_estatisticas = {
    "PLAYER_ID", "PLAYER_NAME", "SEASON",
    "FGA", "FGM", "FG3A", "FG3M"
}

for temporada in temporadas:
    arquivo = pasta_jogadores / f"{temporada}.csv"
    arquivo_novo = not arquivo.exists()

    if arquivo_novo:
        for tentativa in range(1, 4):
            time.sleep(5)

            try:
                resposta = leaguedashplayerstats.LeagueDashPlayerStats(
                    season=temporada,
                    season_type_all_star="Regular Season",
                    per_mode_detailed="Totals",
                    timeout=60
                )

                tabela = resposta.league_dash_player_stats.get_data_frame()
                tabela["SEASON"] = temporada
                break

            except (RequestException, ValueError) as erro:
                print(f"{temporada} | tentativa {tentativa}/3: {erro}")

                if tentativa == 3:
                    raise

                time.sleep(15 * tentativa)
    else:
        tabela = pd.read_csv(arquivo)

    faltantes = colunas_estatisticas - set(tabela.columns)

    if faltantes or tabela.empty:
        raise ValueError(
            f"{temporada}: tabela vazia ou colunas ausentes: {faltantes}"
        )

    ids = pd.to_numeric(tabela["PLAYER_ID"], errors="coerce")

    if not (ids.notna() & ids.gt(0) & ids.mod(1).eq(0)).all():
        raise ValueError(f"{temporada}: identificadores inválidos.")

    if not tabela["SEASON"].eq(temporada).all():
        raise ValueError(f"{temporada}: temporada incorreta ou ausente.")

    if tabela.duplicated(["PLAYER_ID", "SEASON"]).any():
        raise ValueError(f"{temporada}: pares jogador/temporada duplicados.")

    totais = tabela[["FGA", "FGM", "FG3A", "FG3M"]].apply(
        pd.to_numeric, errors="coerce"
    )

    if (
        totais.isna().any().any()
        or totais.lt(0).any().any()
        or totais.mod(1).ne(0).any().any()
    ):
        raise ValueError(f"{temporada}: totais inválidos.")

    if (
        totais["FGM"].gt(totais["FGA"]).any()
        or totais["FG3A"].gt(totais["FGA"]).any()
        or totais["FG3M"].gt(totais["FG3A"]).any()
        or totais["FG3M"].gt(totais["FGM"]).any()
        or (totais["FGM"] - totais["FG3M"]).gt(
            totais["FGA"] - totais["FG3A"]
        ).any()
    ):
        raise ValueError(f"{temporada}: totais inconsistentes.")

    if arquivo_novo:
        temporario = arquivo.with_suffix(".tmp")
        tabela.to_csv(temporario, index=False)
        temporario.replace(arquivo)

    tabelas_jogadores.append(tabela)
    print(f"{temporada}: {len(tabela)} jogadores.")

base_esperada = pd.concat(tabelas_jogadores, ignore_index=True)

print("Total de pares jogador/temporada:", len(base_esperada))

2015-16: 476 jogadores.
2016-17: 486 jogadores.
2017-18: 540 jogadores.
2018-19: 530 jogadores.
2019-20: 529 jogadores.
2020-21: 540 jogadores.
2021-22: 605 jogadores.
2022-23: 539 jogadores.
2023-24: 572 jogadores.
2024-25: 569 jogadores.
Total de pares jogador/temporada: 5386


In [6]:
# MÓDULO 3 — PLANEJAMENTO DA COLETA POR JOGADOR E TEMPORADA
# Organiza os pares únicos que serão consultados.

pares_coleta = (
    base_esperada[["PLAYER_ID", "SEASON"]]
    .drop_duplicates()
    .sort_values(["SEASON", "PLAYER_ID"])
    .reset_index(drop=True)
)

print("Pares planejados:", len(pares_coleta))
display(pares_coleta.groupby("SEASON").size().rename("JOGADORES"))

Pares planejados: 5386


SEASON
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
Name: JOGADORES, dtype: int64

In [13]:
# MÓDULO 4 — COLETA INCREMENTAL DOS ARREMESSOS
# Preserva arquivos existentes, registra tentativas e permite retomada.
# A validade dos arquivos será conferida no módulo 7.

inicio = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
arquivo_log = pasta_arremessos / f"log_{inicio}.csv"

registros = []
falhas_consecutivas = 0

for par in pares_coleta.itertuples(index=False):
    player_id = int(par.PLAYER_ID)
    temporada = par.SEASON

    pasta = pasta_arremessos / temporada
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{player_id}.csv"

    if arquivo.exists():
        continue

    sucesso = False

    for tentativa in range(1, 4):
        time.sleep(1)

        try:
            resposta = shotchartdetail.ShotChartDetail(
                player_id=player_id,
                team_id=0,
                season_nullable=temporada,
                season_type_all_star="Regular Season",
                context_measure_simple="FGA",
                timeout=60
            )

            tabela = resposta.shot_chart_detail.get_data_frame()
            tabela["SEASON"] = temporada

        except (RequestException, ValueError) as erro:
            registros.append({
                "PLAYER_ID": player_id,
                "SEASON": temporada,
                "TENTATIVA": tentativa,
                "STATUS": "falha",
                "LINHAS": None,
                "ERRO": str(erro),
                "DATA_UTC": datetime.now(timezone.utc).isoformat()
            })

            pd.DataFrame(registros).to_csv(arquivo_log, index=False)

            if tentativa < 3:
                time.sleep(15 * tentativa)

            continue

        temporario = arquivo.with_suffix(".tmp")
        tabela.to_csv(temporario, index=False)
        temporario.replace(arquivo)

        registros.append({
            "PLAYER_ID": player_id,
            "SEASON": temporada,
            "TENTATIVA": tentativa,
            "STATUS": "salvo" if len(tabela) else "resposta_vazia",
            "LINHAS": len(tabela),
            "ERRO": "",
            "DATA_UTC": datetime.now(timezone.utc).isoformat()
        })

        pd.DataFrame(registros).to_csv(arquivo_log, index=False)

        print(f"{temporada} | {player_id}: {len(tabela)} arremessos.")
        sucesso = True
        break

    falhas_consecutivas = 0 if sucesso else falhas_consecutivas + 1

    if falhas_consecutivas >= 3:
        print("Três pares consecutivos falharam. Pausa de 15 minutos.")
        time.sleep(15 * 60)
        falhas_consecutivas = 0

print("Coleta encerrada.")

if registros:
    print("Log:", arquivo_log.resolve())
else:
    print("Todos os pares já possuíam arquivos.")

2021-22 | 1628411: 18 arremessos.
2021-22 | 1628415: 526 arremessos.
2021-22 | 1628418: 148 arremessos.
2021-22 | 1628419: 15 arremessos.
2021-22 | 1628420: 773 arremessos.
2021-22 | 1628422: 4 arremessos.
2021-22 | 1629645: 808 arremessos.
2021-22 | 1629646: 47 arremessos.
2021-22 | 1629647: 649 arremessos.
2021-22 | 1629649: 188 arremessos.
2021-22 | 1629650: 108 arremessos.
2021-22 | 1629651: 264 arremessos.
2022-23 | 1630256: 229 arremessos.
2022-23 | 1630264: 132 arremessos.
2022-23 | 1630267: 13 arremessos.
2022-23 | 1630270: 10 arremessos.
2022-23 | 1630284: 107 arremessos.
2022-23 | 1630288: 57 arremessos.
2022-23 | 1631213: 23 arremessos.
2022-23 | 1631214: 0 arremessos.
2022-23 | 1631216: 182 arremessos.
2022-23 | 1631217: 47 arremessos.
2022-23 | 1631219: 53 arremessos.
2022-23 | 1631220: 5 arremessos.
2024-25 | 201572: 774 arremessos.
2024-25 | 201587: 238 arremessos.
2024-25 | 201599: 140 arremessos.
2024-25 | 201935: 1295 arremessos.
2024-25 | 201939: 1258 arremessos.
202

In [8]:
# MÓDULO 5 — COLETA DOS PERFIS FÍSICOS POR TEMPORADA
# Preserva os campos retornados e registra a data das novas consultas.

tabelas_perfis = []

colunas_perfis = {
    "PLAYER_ID", "SEASON",
    "PLAYER_HEIGHT_INCHES", "PLAYER_WEIGHT"
}

for temporada in temporadas:
    arquivo = pasta_perfis / f"{temporada}.csv"
    arquivo_novo = not arquivo.exists()

    if arquivo_novo:
        for tentativa in range(1, 4):
            time.sleep(5)

            try:
                resposta = (
                    leaguedashplayerbiostats.LeagueDashPlayerBioStats(
                        season=temporada,
                        season_type_all_star="Regular Season",
                        timeout=60
                    )
                )

                tabela = (
                    resposta.league_dash_player_bio_stats.get_data_frame()
                )

                tabela["SEASON"] = temporada
                tabela["BIO_COLLECTED_AT"] = (
                    datetime.now(timezone.utc).isoformat()
                )
                break

            except (RequestException, ValueError) as erro:
                print(f"{temporada} | tentativa {tentativa}/3: {erro}")

                if tentativa == 3:
                    raise

                time.sleep(15 * tentativa)
    else:
        tabela = pd.read_csv(arquivo)

    faltantes = colunas_perfis - set(tabela.columns)

    if faltantes or tabela.empty:
        raise ValueError(
            f"{temporada}: tabela vazia ou colunas ausentes: {faltantes}"
        )

    if arquivo_novo:
        temporario = arquivo.with_suffix(".tmp")
        tabela.to_csv(temporario, index=False)
        temporario.replace(arquivo)

    tabelas_perfis.append(tabela)
    print(f"{temporada}: {len(tabela)} perfis.")

df_perfis = pd.concat(tabelas_perfis, ignore_index=True)

2015-16: 476 perfis.
2016-17: 486 perfis.
2017-18: 540 perfis.
2018-19: 530 perfis.
2019-20: 529 perfis.
2020-21: 540 perfis.
2021-22: 605 perfis.
2022-23: 539 perfis.
2023-24: 572 perfis.
2024-25: 569 perfis.


In [14]:
# MÓDULO 6 — INVENTÁRIO E IDENTIFICAÇÃO DE ARQUIVOS EXTRAS
# Lista os CSVs de eventos encontrados e identifica caminhos não previstos.
# O conteúdo dos arquivos esperados será auditado no módulo seguinte.

caminhos_esperados = {
    (pasta_arremessos / par.SEASON / f"{int(par.PLAYER_ID)}.csv").resolve()
    for par in pares_coleta.itertuples(index=False)
}

caminhos_encontrados = {
    arquivo.resolve()
    for arquivo in pasta_arremessos.glob("*/*.csv")
}

arquivos_extras = pd.DataFrame(
    [
        {"ARQUIVO": str(arquivo)}
        for arquivo in sorted(caminhos_encontrados - caminhos_esperados)
    ],
    columns=["ARQUIVO"]
)

print("Arquivos esperados:", len(caminhos_esperados))
print("Arquivos encontrados:", len(caminhos_encontrados))
print("Arquivos ausentes:", len(caminhos_esperados - caminhos_encontrados))
print("Arquivos extras:", len(arquivos_extras))

display(arquivos_extras)

Arquivos esperados: 5386
Arquivos encontrados: 5386
Arquivos ausentes: 0
Arquivos extras: 0


,ARQUIVO


In [ ]:
# MÓDULO 7 — AUDITORIA DOS ARREMESSOS E DOS ARQUIVOS VAZIOS
# Confere identidade, estrutura, duplicatas, flags e totais.
# CSV vazio só é aprovado quando seus totais conferem com a referência.

resultados = []

colunas_arremessos = [
    "PLAYER_ID", "SEASON", "GAME_ID", "GAME_EVENT_ID",
    "SHOT_ATTEMPTED_FLAG", "SHOT_MADE_FLAG", "SHOT_TYPE",
    "SHOT_DISTANCE", "LOC_X", "LOC_Y"
]

for jogador in base_esperada.itertuples(index=False):
    player_id = int(jogador.PLAYER_ID)
    temporada = jogador.SEASON
    arquivo = pasta_arremessos / temporada / f"{player_id}.csv"

    resultado = {
        "PLAYER_ID": player_id,
        "SEASON": temporada,
        "FGA_ESPERADO": jogador.FGA,
        "LINHAS_SALVAS": None,
        "STATUS": "",
        "DETALHE": ""
    }

    if not arquivo.is_file():
        resultado["STATUS"] = "arquivo_ausente"

    else:
        try:
            tabela = pd.read_csv(
                arquivo,
                dtype={"GAME_ID": "string"},
                usecols=colunas_arremessos
            )

            resultado["LINHAS_SALVAS"] = len(tabela)
            problemas = []
            chaves = ["GAME_ID", "GAME_EVENT_ID"]

            if tabela[chaves].isna().any().any():
                problemas.append("chave de evento ausente")

            if tabela.duplicated(chaves).any():
                problemas.append("eventos duplicados")

            if not tabela["PLAYER_ID"].eq(player_id).all():
                problemas.append("jogador incorreto ou ausente")

            if not tabela["SEASON"].eq(temporada).all():
                problemas.append("temporada incorreta ou ausente")

            if not tabela["SHOT_ATTEMPTED_FLAG"].eq(1).all():
                problemas.append("flag de tentativa inválida")

            if not tabela["SHOT_MADE_FLAG"].isin([0, 1]).all():
                problemas.append("flag de resultado inválida")

            if not tabela["SHOT_TYPE"].isin(
                ["2PT Field Goal", "3PT Field Goal"]
            ).all():
                problemas.append("tipo de arremesso inválido")

            acertos = pd.to_numeric(
                tabela["SHOT_MADE_FLAG"], errors="coerce"
            )

            tres_pontos = tabela["SHOT_TYPE"].eq("3PT Field Goal")

            comparacoes = {
                "FGA": (len(tabela), jogador.FGA),
                "FGM": (acertos.sum(), jogador.FGM),
                "FG3A": (int(tres_pontos.sum()), jogador.FG3A),
                "FG3M": (acertos.loc[tres_pontos].sum(), jogador.FG3M)
            }

            for nome, (observado, esperado) in comparacoes.items():
                resultado[f"{nome}_OBSERVADO"] = observado
                resultado[f"{nome}_ESPERADO"] = esperado

                if observado != esperado:
                    problemas.append(f"{nome} diferente")

            resultado["STATUS"] = "verificar" if problemas else "confere"
            resultado["DETALHE"] = "; ".join(problemas)

        except Exception as erro:
            resultado["STATUS"] = "erro_leitura_ou_estrutura"
            resultado["DETALHE"] = str(erro)

    resultados.append(resultado)

auditoria_arremessos = pd.DataFrame(resultados)
auditoria_arremessos["AUDITADO_EM_UTC"] = (
    datetime.now(timezone.utc).isoformat()
)

pendencias_arremessos = auditoria_arremessos.loc[
    auditoria_arremessos["STATUS"].ne("confere")
].copy()

display(pd.crosstab(
    auditoria_arremessos["SEASON"],
    auditoria_arremessos["STATUS"]
))

print("Pares com pendências:", len(pendencias_arremessos))
display(pendencias_arremessos)

print("Verificação dos arquivos vazios:")
display(auditoria_arremessos.loc[
    auditoria_arremessos["LINHAS_SALVAS"].eq(0),
    ["SEASON", "PLAYER_ID", "FGA_ESPERADO", "STATUS", "DETALHE"]
])

STATUS,confere
SEASON,
2015-16,476
2016-17,486
2017-18,540
2018-19,530
2019-20,529
2020-21,540
2021-22,605
2022-23,539
2023-24,572


Pares com pendências: 0


,PLAYER_ID,SEASON,FGA_ESPERADO,LINHAS_SALVAS,STATUS,DETALHE,FGA_OBSERVADO,FGM_OBSERVADO,FGM_ESPERADO,FG3A_OBSERVADO,FG3A_ESPERADO,FG3M_OBSERVADO,FG3M_ESPERADO,AUDITADO_EM_UTC


Verificação dos arquivos vazios:


,SEASON,PLAYER_ID,FGA_ESPERADO,STATUS,DETALHE
403,2015-16,1626155,0,confere,
572,2016-17,1627863,0,confere,
1001,2017-18,1628500,0,confere,
1461,2017-18,203590,0,confere,
1469,2017-18,1628399,0,confere,
1678,2018-19,1628994,0,confere,
1763,2018-19,204066,0,confere,
1815,2018-19,1628424,0,confere,
2001,2018-19,1627755,0,confere,
2233,2019-20,1629122,0,confere,


In [18]:
# MÓDULO 8 — AUDITORIA DOS PERFIS FÍSICOS
# Confere cobertura, chaves, duplicatas e valores ausentes ou não positivos.
# Não remove nem preenche dados; registra os problemas encontrados.

resultados_perfis = []
detalhes_perfis = []

for temporada in temporadas:
    arquivo = pasta_perfis / f"{temporada}.csv"

    esperados = set(
        base_esperada.loc[
            base_esperada["SEASON"].eq(temporada), "PLAYER_ID"
        ].astype(int)
    )

    resultado = {
        "SEASON": temporada,
        "ESPERADOS": len(esperados),
        "ENCONTRADOS": None,
        "FALTANTES": None,
        "EXTRAS": None,
        "STATUS": ""
    }

    try:
        tabela = pd.read_csv(arquivo)

        faltantes_colunas = colunas_perfis - set(tabela.columns)

        if faltantes_colunas or tabela.empty:
            raise ValueError(
                f"Tabela vazia ou colunas ausentes: {faltantes_colunas}"
            )

        ids = pd.to_numeric(tabela["PLAYER_ID"], errors="coerce")
        ids_validos = ids.notna() & ids.gt(0) & ids.mod(1).eq(0)
        temporada_correta = tabela["SEASON"].eq(temporada)

        recebidos = set(
            ids.loc[ids_validos & temporada_correta].astype(int)
        )

        faltantes = esperados - recebidos
        extras = recebidos - esperados

        altura = pd.to_numeric(
            tabela["PLAYER_HEIGHT_INCHES"], errors="coerce"
        )
        peso = pd.to_numeric(tabela["PLAYER_WEIGHT"], errors="coerce")

        verificacoes = [
            (~ids_validos, "identificador inválido"),
            (~temporada_correta, "temporada incorreta ou ausente"),
            (
                tabela.duplicated(["PLAYER_ID", "SEASON"], keep=False),
                "perfil duplicado"
            ),
            (
                altura.isna() | altura.le(0) | altura.isin([float("inf")]),
                "altura ausente ou inválida"
            ),
            (
                peso.isna() | peso.le(0) | peso.isin([float("inf")]),
                "peso ausente ou inválido"
            )
        ]

        tem_problemas = bool(faltantes or extras)

        for mascara, motivo in verificacoes:
            if mascara.any():
                tem_problemas = True

                for player_id in tabela.loc[mascara, "PLAYER_ID"]:
                    detalhes_perfis.append({
                        "SEASON": temporada,
                        "PLAYER_ID": player_id,
                        "PROBLEMA": motivo
                    })

        for conjunto, motivo in [
            (faltantes, "jogador esperado sem perfil"),
            (extras, "perfil fora da lista esperada")
        ]:
            for player_id in sorted(conjunto):
                detalhes_perfis.append({
                    "SEASON": temporada,
                    "PLAYER_ID": player_id,
                    "PROBLEMA": motivo
                })

        resultado.update({
            "ENCONTRADOS": len(esperados & recebidos),
            "FALTANTES": len(faltantes),
            "EXTRAS": len(extras),
            "STATUS": "verificar" if tem_problemas else "confere"
        })

    except Exception as erro:
        resultado["STATUS"] = "erro_leitura_ou_estrutura"
        detalhes_perfis.append({
            "SEASON": temporada,
            "PLAYER_ID": None,
            "PROBLEMA": str(erro)
        })

    resultados_perfis.append(resultado)

auditoria_perfis = pd.DataFrame(resultados_perfis)
auditoria_perfis["AUDITADO_EM_UTC"] = (
    datetime.now(timezone.utc).isoformat()
)

pendencias_perfis = pd.DataFrame(
    detalhes_perfis,
    columns=["SEASON", "PLAYER_ID", "PROBLEMA"]
)

display(auditoria_perfis)
print("Ocorrências para investigar:", len(pendencias_perfis))
display(pendencias_perfis)

,SEASON,ESPERADOS,ENCONTRADOS,FALTANTES,EXTRAS,STATUS,AUDITADO_EM_UTC
0,2015-16,476,476,0,0,confere,2026-10-23T18:10:27.692133+00:00
1,2016-17,486,486,0,0,confere,2026-10-23T18:10:27.692133+00:00
2,2017-18,540,540,0,0,confere,2026-10-23T18:10:27.692133+00:00
3,2018-19,530,530,0,0,confere,2026-10-23T18:10:27.692133+00:00
4,2019-20,529,529,0,0,confere,2026-10-23T18:10:27.692133+00:00
5,2020-21,540,540,0,0,confere,2026-10-23T18:10:27.692133+00:00
6,2021-22,605,605,0,0,confere,2026-10-23T18:10:27.692133+00:00
7,2022-23,539,539,0,0,confere,2026-10-23T18:10:27.692133+00:00
8,2023-24,572,572,0,0,confere,2026-10-23T18:10:27.692133+00:00
9,2024-25,569,569,0,0,confere,2026-10-23T18:10:27.692133+00:00


Ocorrências para investigar: 0


,SEASON,PLAYER_ID,PROBLEMA


In [19]:
# MÓDULO 9 — CONSOLIDAÇÃO E SALVAMENTO DAS EVIDÊNCIAS DA AUDITORIA
# Salva os resultados já calculados, sem repetir consultas à API.
# Execute novamente os módulos 6 a 8 após qualquer alteração nos dados.
# A aprovação se limita às verificações implementadas.

comparacao = pares_coleta.merge(
    auditoria_arremessos[["PLAYER_ID", "SEASON"]],
    on=["PLAYER_ID", "SEASON"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

if comparacao["_merge"].ne("both").any():
    raise ValueError("A auditoria não cobre exatamente os pares planejados.")

if (
    auditoria_perfis["SEASON"].duplicated().any()
    or set(auditoria_perfis["SEASON"]) != set(temporadas)
):
    raise ValueError("A auditoria de perfis não cobre as temporadas.")

status_validos = [
    "confere", "arquivo_ausente",
    "verificar", "erro_leitura_ou_estrutura"
]

if not auditoria_arremessos["STATUS"].isin(status_validos).all():
    raise ValueError("Status inesperado na auditoria de arremessos.")

if not auditoria_perfis["STATUS"].isin(
    ["confere", "verificar", "erro_leitura_ou_estrutura"]
).all():
    raise ValueError("Status inesperado na auditoria de perfis.")

resumo_temporadas = pd.crosstab(
    auditoria_arremessos["SEASON"],
    auditoria_arremessos["STATUS"]
).reindex(columns=status_validos, fill_value=0)

resumo_temporadas = resumo_temporadas.rename(columns={
    "confere": "PARES_APROVADOS",
    "arquivo_ausente": "ARQUIVOS_AUSENTES",
    "verificar": "PARES_DIVERGENTES",
    "erro_leitura_ou_estrutura": "ERROS_LEITURA_OU_ESTRUTURA"
})

resumo_temporadas["PARES_ESPERADOS"] = (
    pares_coleta.groupby("SEASON").size()
)

resumo_temporadas = resumo_temporadas.reset_index()
resumo_temporadas.columns.name = None

resumo_temporadas = resumo_temporadas.merge(
    auditoria_perfis[["SEASON", "STATUS"]].rename(
        columns={"STATUS": "STATUS_PERFIS"}
    ),
    on="SEASON",
    validate="one_to_one"
)

pendencias_arremessos = auditoria_arremessos.loc[
    auditoria_arremessos["STATUS"].ne("confere")
].copy()

sem_pendencias = (
    pendencias_arremessos.empty
    and auditoria_perfis["STATUS"].eq("confere").all()
    and pendencias_perfis.empty
    and arquivos_extras.empty
)

momento = datetime.now(timezone.utc)

resumo_geral = pd.DataFrame([{
    "PARES_ESPERADOS": len(pares_coleta),
    "PARES_APROVADOS": int(
        auditoria_arremessos["STATUS"].eq("confere").sum()
    ),
    "ARQUIVOS_AUSENTES": int(
        auditoria_arremessos["STATUS"].eq("arquivo_ausente").sum()
    ),
    "PARES_DIVERGENTES": int(
        auditoria_arremessos["STATUS"].eq("verificar").sum()
    ),
    "ERROS_LEITURA_OU_ESTRUTURA": int(
        auditoria_arremessos["STATUS"]
        .eq("erro_leitura_ou_estrutura").sum()
    ),
    "ARQUIVOS_EXTRAS": len(arquivos_extras),
    "TEMPORADAS_PERFIS_COM_PENDENCIAS": int(
        auditoria_perfis["STATUS"].ne("confere").sum()
    ),
    "OCORRENCIAS_PENDENCIAS_PERFIS": len(pendencias_perfis),
    "RESULTADO": (
        "sem pendências nas verificações executadas"
        if sem_pendencias
        else "existem pendências para investigar"
    )
}])

pasta_resultados = (
    pasta_auditorias / momento.strftime("%Y%m%dT%H%M%S%fZ")
)
pasta_resultados.mkdir(parents=True, exist_ok=False)

tabelas_exportacao = {
    "resumo_geral.csv": resumo_geral,
    "resumo_por_temporada.csv": resumo_temporadas,
    "auditoria_arremessos.csv": auditoria_arremessos,
    "pendencias_arremessos.csv": pendencias_arremessos,
    "auditoria_perfis.csv": auditoria_perfis,
    "pendencias_perfis.csv": pendencias_perfis,
    "arquivos_extras.csv": arquivos_extras
}

for nome, tabela in tabelas_exportacao.items():
    exportacao = tabela.copy()
    exportacao["CONSOLIDADO_EM_UTC"] = momento.isoformat()
    exportacao.to_csv(
        pasta_resultados / nome,
        index=False,
        encoding="utf-8-sig"
    )

display(resumo_geral.T.rename(columns={0: "VALOR"}))
display(resumo_temporadas)

print("Resultados salvos em:", pasta_resultados.resolve())

,VALOR
PARES_ESPERADOS,5386
PARES_APROVADOS,5386
ARQUIVOS_AUSENTES,0
PARES_DIVERGENTES,0
ERROS_LEITURA_OU_ESTRUTURA,0
ARQUIVOS_EXTRAS,0
TEMPORADAS_PERFIS_COM_PENDENCIAS,0
OCORRENCIAS_PENDENCIAS_PERFIS,0
RESULTADO,sem pendências nas verificações executadas


,SEASON,PARES_APROVADOS,ARQUIVOS_AUSENTES,PARES_DIVERGENTES,ERROS_LEITURA_OU_ESTRUTURA,PARES_ESPERADOS,STATUS_PERFIS
0,2015-16,476,0,0,0,476,confere
1,2016-17,486,0,0,0,486,confere
2,2017-18,540,0,0,0,540,confere
3,2018-19,530,0,0,0,530,confere
4,2019-20,529,0,0,0,529,confere
5,2020-21,540,0,0,0,540,confere
6,2021-22,605,0,0,0,605,confere
7,2022-23,539,0,0,0,539,confere
8,2023-24,572,0,0,0,572,confere
9,2024-25,569,0,0,0,569,confere


Resultados salvos em: C:\Users\felip\Desktop\teste-nba\dados\auditorias\20261023T181053808012Z
